# LoadWatch — Exploratory Data Analysis

This notebook documents methodology checks for judges: workload distributions, label imbalance, time-based split sanity, and simple associations between workload features and soft-tissue labels.

**Run `python scripts/run_pipeline.py` first** so `data/processed/` is populated.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path('..').resolve()
PROCESSED = ROOT / 'data' / 'processed'
MODELS = ROOT / 'models'
SEASON = '2023-24'

sns.set_theme(style='whitegrid', context='notebook')

In [ ]:
features = pd.read_parquet(PROCESSED / f'features_{SEASON}.parquet')
scored = pd.read_parquet(PROCESSED / f'scored_{SEASON}.parquet')
injuries = pd.read_parquet(PROCESSED / f'injuries_clean_{SEASON}.parquet')
metrics = json.loads((MODELS / 'metrics.json').read_text())

print(features.shape, 'positive rate', features['label'].mean())
print('players', features['PLAYER_ID'].nunique())
display(metrics)

## Label imbalance

Soft-tissue events within a 7-day horizon are rare. Accuracy alone is misleading; prefer PR-AUC.

In [ ]:
ax = features['label'].value_counts().sort_index().plot(kind='bar', color=['#1a5f7a', '#c0392b'])
ax.set_xticklabels(['No injury (0)', 'Soft-tissue in 7d (1)'], rotation=0)
ax.set_ylabel('Player-games')
ax.set_title('Label distribution')
plt.show()

## Workload feature distributions

In [ ]:
cols = ['minutes_last_7d', 'minutes_last_14d', 'games_last_7d', 'rest_days', 'age', 'days_since_last_injury']
cols = [c for c in cols if c in features.columns]
features[cols].hist(bins=30, figsize=(12, 8), color='#1a5f7a')
plt.suptitle('Feature histograms')
plt.tight_layout()
plt.show()

## Association: back-to-backs vs label rate

In [ ]:
tab = features.groupby('is_back_to_back')['label'].agg(['mean', 'count'])
tab.index = tab.index.map({0: 'Not B2B', 1: 'Back-to-back'})
display(tab)
tab['mean'].plot(kind='bar', color=['#1a5f7a', '#c0392b'], title='Empirical soft-tissue label rate')
plt.ylabel('Rate')
plt.show()

## Time-based split check

Training uses earlier games; testing uses later games — no random shuffle leakage.

In [ ]:
print('Train ends', metrics.get('train_end_date'), '| Test starts', metrics.get('test_start_date'))
pd.DataFrame(metrics.get('metrics', []))

## Risk score vs eventual labels (descriptive only)

In [ ]:
scored['risk_decile'] = pd.qcut(scored['risk_score'], 10, labels=False, duplicates='drop')
rate = scored.groupby('risk_decile')['label'].mean()
rate.plot(marker='o', color='#1a5f7a', title='Label rate by risk-score decile')
plt.xlabel('Risk decile (low → high)')
plt.ylabel('Soft-tissue label rate')
plt.show()

## Global SHAP importance

In [ ]:
imp_path = MODELS / 'shap_global_importance.csv'
if imp_path.exists():
    imp = pd.read_csv(imp_path, index_col=0)
    imp.head(10).plot(kind='barh', legend=False, color='#143d5c')
    plt.gca().invert_yaxis()
    plt.title('Mean |SHAP|')
    plt.show()
else:
    print('Run pipeline with SHAP enabled first.')